In [1]:
import numpy as np
import pygame
from scipy.integrate import solve_bvp
import multiprocessing
multiprocessing.set_start_method('spawn' , force = True)
import os
from stable_baselines3 import PPO ,A2C , DDPG ,TD3 , SAC
from stable_baselines3.common.vec_env import DummyVecEnv , SubprocVecEnv
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.evaluation import evaluate_policy
import time
import gymnasium as gym
from gymnasium.spaces import Discrete, Box, Dict
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
import math
from stable_baselines3.common.callbacks import EvalCallback
import optuna

pygame 2.5.2 (SDL 2.28.3, Python 3.11.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
import sys
print(sys.version)

3.11.7 | packaged by Anaconda, Inc. | (main, Dec 15 2023, 18:05:47) [MSC v.1916 64 bit (AMD64)]


In [3]:
class Elastica_env(gym.Env):
    def __init__(self):
        super(Elastica_env, self).__init__()
        self.action_space = Box(low=np.array([-0.2, -0.2], dtype=np.float32), high=np.array([0.1 , 0.2], dtype=np.float64))
        self.observation_space = Box(low=np.float32(-100), high=np.float32(100), shape=(13,), dtype=np.float64)
        self.target = Box(low=np.array([4.5, -1], dtype=np.float32), high=np.array([5.57, 1], dtype=np.float64))
        self.num_timestep = 0
        self.reward = 0
        self.x = []
        self.y = []
        self.screen_width = 800.0
        self.screen_height = 600.0
        self.zoom_factor = 60.0
        self.enable_render = False
        self.h = -0.4
        self.v = 0.15

    # Add the seed() method
    def seed(self, seed=None):
        self.np_random, seed = gym.utils.seeding.np_random(seed)
        return seed
    def step(self, action):
        self.num_timestep += 1
        self.h += action[0]
        self.v += action[1]
        self.X, self.Y, self.theta_dash_0  ,self.theta_dash_l, self.theta_l , self.E = self.elastica(self.h, self.v)

        new_observation = self.get_observation()
        self.render(self.enable_render)
        self.reward = self.score()
        done = self.get_done()
        truncation = self.get_truncation()
        info = {}
        return new_observation, self.reward, done, truncation, info

    def elastica(self, h, v):
        l = 6
        s = np.linspace(0, l, 500)
        def f(s, y):
            return np.vstack((y[1], h * np.sin(y[0]) - v * np.cos(y[0])))
        def bc(ya, yb):
            return np.array([ya[0] - 0, yb[1] - 0])
        y0 = np.zeros((2, s.size))
        sol = solve_bvp(f, bc, s, y0)
        theta = sol.sol(s)[0]
        dtheta_ds = sol.sol(s)[1]
        y1 = np.cos(theta)
        y2 = np.sin(theta)
        y3 = (dtheta_ds)**2
        self.x = []
        self.y = []
        for i in range(len(s)):
            self.x.append(np.trapz(y1[:i+1], x=s[:i+1]))
            self.y.append(np.trapz(y2[:i+1], x=s[:i+1]))

        e = 0.5*(np.trapz(y3 , s))-v*(np.trapz(y2 , s)) + h*(np.trapz(1-y1 , s))

        return self.x, self.y, dtheta_ds[0] ,dtheta_ds[-1] , theta[-1] , e 

    def get_observation(self):
        self.x_tip = self.X[-1]
        self.y_tip = self.Y[-1]
        d = ((self.x_tip - self.x_target)**2 + (self.y_tip - self.y_target)**2)**0.5
        return np.array([self.x_tip, self.y_tip, self.X[200], self.Y[200], self.X[400], self.Y[400], 
                         self.theta_l , self.theta_dash_0 , self.theta_dash_l ,self.E  ,
                         self.x_target, self.y_target ,d] ,  dtype=np.float64)

    def score(self):
        d = ((self.x_tip - self.x_target)**2 + (self.y_tip - self.y_target)**2)**0.5
        #d_score = -(d)**2
        d_score = math.exp(-d)
#         if d>0 and d<0.002:
#             d_score += 1
#         elif d>0 and d<0.05:
#             d_score += 0.3
#         elif d>0 and d>0.1:
#             d_score += 0.1
        

        total_score = d_score
        return total_score

    def get_done(self):
        done = False
        d = ((self.x_tip - self.x_target)**2 + (self.y_tip - self.y_target)**2)**0.5
        if d < 0.002:
            done = True
        return done

    def get_truncation(self):
        truncation = False
        if self.num_timestep > 19 :
            truncation = True
        return truncation

    def reset(self, seed=None):
        if seed is not None:
            self.np_random, seed = gym.utils.seeding.np_random(seed)
        targ = self.target.sample()
        self.x_target = targ[0]
        self.y_target = targ[1]
        if self.y_target<=0:
            self.h = -0.4
            self.v = 0.15
        if self.y_target>0:
            self.h = -0.4
            self.v = -0.15

        self.X, self.Y, self.theta_dash_0 ,self.theta_dash_l , self.theta_l , self.E  = self.elastica(self.h, self.v)
        self.num_timestep = 0
        self.reward = 0
        observation = self.get_observation()
        info = {}
        return observation, info

    def render(self, enable_render):
        if not enable_render:
            return
        pygame.init()
        screen = pygame.display.set_mode((int(self.screen_width), int(self.screen_height)))
        pygame.display.set_caption("Elastica")
        screen.fill((255, 255, 255))
        offset_x = (self.screen_width - 10 * self.zoom_factor) / 2
        offset_y = (self.screen_height - 1.5 * self.zoom_factor) / 2
        points = [(self.X[i], self.Y[i]) for i in range(len(self.X))]
        pygame.draw.lines(screen, (0, 0, 0), False, [(point[0] * self.zoom_factor + offset_x, point[1] * self.zoom_factor + offset_y) for point in points])
        pygame.draw.line(screen, (255, 0, 0), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y), ((self.X[-1]) * self.zoom_factor + offset_x + 50 * np.sign(self.h), (self.Y[-1]) * self.zoom_factor + offset_y), 3)
        pygame.draw.line(screen, (0, 255, 0), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y), ((self.X[-1]) * self.zoom_factor + offset_x, (self.Y[-1]) * self.zoom_factor + offset_y + 50 * np.sign(self.v)), 3)
        pygame.draw.line(screen, (0, 0, 0), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y + 25), 3)
        pygame.draw.line(screen, (0, 0, 0), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y), ((self.X[0]) * self.zoom_factor + offset_x, (self.Y[0]) * self.zoom_factor + offset_y - 25), 3)
        pygame.draw.circle(screen, (255, 0, 0), (int(self.x_target * self.zoom_factor + offset_x), int(self.y_target * self.zoom_factor + offset_y)), 5)
        font = pygame.font.Font(None, 36)
        score_text = font.render(f"Timesteps: {self.num_timestep}", True, (0, 0, 0))
        screen.blit(score_text, (int(self.screen_width - score_text.get_width() - 30), 120))
        pygame.display.flip()

    def close(self):
        pygame.quit()


In [4]:
import pickle

env = Elastica_env()
try:
    pickle.dumps(env)
    print("Environment is serializable")
except Exception as e:
    print("Environment is not serializable:", e)


Environment is serializable


In [5]:
num_cpu_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores available: {num_cpu_cores}")

Number of CPU cores available: 4


In [6]:
env = Elastica_env()

In [7]:
check_env(env)
env.close()

C:\Users\Jayhind\anaconda3\Lib\site-packages\stable_baselines3\common\env_checker.py:453: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(


In [8]:
env.enable_render = True
episodes = 10
for episode in range(1, episodes+1):
    state , info = env.reset()
    done = False
    score = 0
    truncation=False
    while not (done or truncation):
        #env.render(True )
        action = env.action_space.sample()
        #print(action)
        n_state, reward, done , truncation ,info = env.step(action)
        score+=reward
    print('Episode:{} Score:{}'.format(episode, score))
env.close()

Episode:1 Score:5.2037539716652725
Episode:2 Score:2.9841496345236243
Episode:3 Score:3.041504108332487
Episode:4 Score:1.1177812737419086
Episode:5 Score:4.384909683257358
Episode:6 Score:4.235943357330951
Episode:7 Score:3.910784076557725
Episode:8 Score:4.309556326039286
Episode:9 Score:2.203898928812314
Episode:10 Score:3.6390781108612384


In [74]:
# Function to create and seed each environment
def make_custom_env(rank: int, seed: int = 0):
    def _init():
        env = Elastica_env()  
        env.seed(seed + rank)
        return env
    return _init

# Set the multiprocessing start method (recommended on Windows)
multiprocessing.set_start_method('spawn', force=True)

# Set up the number of environments
num_cpu = 3  # Number of environments to run in parallel (number of CPU cores)
env2 = SubprocVecEnv([make_custom_env(i) for i in range(num_cpu)], start_method='spawn')
eval_env1 = SubprocVecEnv([make_custom_env(i) for i in range(num_cpu)], start_method='spawn')


# Initialize and train the RL agent 
# model = SAC("MlpPolicy", env2, verbose=1)
# model.learn(total_timesteps=200)


In [ ]:
env.enable_render = False
log_path = os.path.join('Training' , 'Logs')
os.makedirs(os.path.dirname(log_path) , exist_ok = True)
eval_env = Elastica_env()
eval_env = Monitor(eval_env , log_path)
#policy_kwargs = dict(net_arch=dict(pi=[256, 256 , 256], qf=[500, 400 , 400]) )  # by default tanh activation function
#policy_kwargs = dict(activation_fn=th.nn.ReLU,net_arch=dict(pi=[32, 32], qf=[32, 32]))    # ReLu activation function

def objective(trial):
    learning_rate = trial.suggest_float('learning_rate' , 1e-5 , 1e-3 ,log = True)
    entropy_coef = trial.suggest_float ('entropy_coef' , 1e-4 , 1e-2  , log = True)
    gamma = trial.suggest_float('gamma' ,0.9 ,0.9999 , log = True)
    tau = trial.suggest_float('tau' , 0.005 , 0.05 , log = True)
    batch_size = trial.suggest_categorical('batch_size' , [64 , 128 , 256 , 1000 , 2000 , 4000 , 8000 , 16000])
    target_update_interval = trial.suggest_categorical('target_update_interval' , [1000 , 5000 , 10000])
    gradient_steps = trial.suggest_categorical('gradient_steps' , [1 , 2, 4])
    buffer_size = trial.suggest_categorical('buffer_size' , [100000 , 200000 , 500000 , 1000000 , 2000000])
    
    model = SAC('MlpPolicy', env2, verbose=0 , learning_rate = learning_rate ,
                ent_coef = entropy_coef , gamma = gamma , tau = tau , batch_size = batch_size , 
                target_update_interval = target_update_interval , gradient_steps= gradient_steps ,
                buffer_size = buffer_size )
    
    eval_callback = EvalCallback(eval_env ,  eval_freq = 100 ,deterministic = True )
    model.learn(total_timesteps = 200 , callback = eval_callback)

    mean_reward , _ = evaluate_policy(model , eval_env , n_eval_episodes = 10 ,deterministic=True)
    return mean_reward

study = optuna.create_study (direction = 'maximize')
study.optimize(objective , n_trials = 50 , n_jobs = 1)
best_parameters = study.best_params
print(best_parameters)

In [76]:
log_path = os.path.join('Training', 'Logs')
#PPO_path = os.path.join('Training', 'Saved Models', 'DDPG')
#print(PPO_path)
#env = DummyVecEnv([lambda: env])
model = SAC( 'MlpPolicy', env2, verbose=0 ,  learning_rate=best_parameters['learning_rate'],
            ent_coef=best_parameters['entropy_coef'], gamma=best_parameters['gamma'], tau=best_parameters['tau'], 
           batch_size=best_parameters['batch_size'], target_update_interval = best_parameters['target_update_interval'],
           gradient_steps=best_parameters['gradient_steps'],buffer_size=best_parameters['buffer_size'])  

In [77]:
eval_callback = EvalCallback(eval_env,  eval_freq = 100 ,deterministic = True )
model.learn(total_timesteps=2000 , callback = eval_callback )

C:\Users\Jayhind\anaconda3\Lib\site-packages\stable_baselines3\common\callbacks.py:414: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x0000026A0069FED0> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x0000026A6BB76350>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


Eval num_timesteps=300, episode_reward=8.46 +/- 1.25
Episode length: 20.00 +/- 0.00
New best mean reward!
Eval num_timesteps=600, episode_reward=7.27 +/- 0.47
Episode length: 20.00 +/- 0.00
Eval num_timesteps=900, episode_reward=7.44 +/- 1.81
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1200, episode_reward=9.32 +/- 1.16
Episode length: 20.00 +/- 0.00
New best mean reward!
Eval num_timesteps=1500, episode_reward=9.25 +/- 3.51
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1800, episode_reward=11.69 +/- 2.02
Episode length: 20.00 +/- 0.00
New best mean reward!


Eval num_timesteps=748500, episode_reward=19.64 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=749000, episode_reward=19.56 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=749500, episode_reward=19.56 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=750000, episode_reward=19.54 +/- 0.10
Episode length: 20.00 +/- 0.00
Eval num_timesteps=750500, episode_reward=19.43 +/- 0.16
Episode length: 20.00 +/- 0.00
Eval num_timesteps=751000, episode_reward=19.42 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=751500, episode_reward=19.51 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=752000, episode_reward=19.46 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=752500, episode_reward=19.60 +/- 0.03
Episode length: 20.00 +/- 0.00
Eval num_timesteps=753000, episode_reward=19.63 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=753500, episode_reward=19.43 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=754000, episo

Eval num_timesteps=795500, episode_reward=19.64 +/- 0.02
Episode length: 20.00 +/- 0.00
Eval num_timesteps=796000, episode_reward=19.42 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=796500, episode_reward=19.41 +/- 0.13
Episode length: 20.00 +/- 0.00
Eval num_timesteps=797000, episode_reward=19.58 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=797500, episode_reward=19.46 +/- 0.03
Episode length: 20.00 +/- 0.00
Eval num_timesteps=798000, episode_reward=19.50 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=798500, episode_reward=19.57 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=799000, episode_reward=19.51 +/- 0.03
Episode length: 20.00 +/- 0.00
Eval num_timesteps=799500, episode_reward=19.37 +/- 0.13
Episode length: 20.00 +/- 0.00
Eval num_timesteps=800000, episode_reward=19.53 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=800500, episode_reward=19.57 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=801000, episo

Eval num_timesteps=842000, episode_reward=19.51 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=842500, episode_reward=19.45 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=843000, episode_reward=19.48 +/- 0.12
Episode length: 20.00 +/- 0.00
Eval num_timesteps=843500, episode_reward=19.49 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=844000, episode_reward=19.47 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=844500, episode_reward=19.52 +/- 0.02
Episode length: 20.00 +/- 0.00
Eval num_timesteps=845000, episode_reward=19.51 +/- 0.03
Episode length: 20.00 +/- 0.00
Eval num_timesteps=845500, episode_reward=19.58 +/- 0.09
Episode length: 20.00 +/- 0.00
Eval num_timesteps=846000, episode_reward=19.46 +/- 0.08
Episode length: 20.00 +/- 0.00
Eval num_timesteps=846500, episode_reward=19.38 +/- 0.22
Episode length: 20.00 +/- 0.00
Eval num_timesteps=847000, episode_reward=19.61 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=847500, episo

...

Eval num_timesteps=1075500, episode_reward=19.48 +/- 0.10
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1076000, episode_reward=19.39 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1076500, episode_reward=19.48 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1077000, episode_reward=19.59 +/- 0.10
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1077500, episode_reward=19.54 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1078000, episode_reward=19.48 +/- 0.11
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1078500, episode_reward=19.63 +/- 0.08
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1079000, episode_reward=19.61 +/- 0.09
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1079500, episode_reward=19.59 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1080000, episode_reward=19.61 +/- 0.11
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1080500, episode_reward=19.59 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=10

Eval num_timesteps=1122000, episode_reward=19.58 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1122500, episode_reward=19.59 +/- 0.02
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1123000, episode_reward=19.59 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1123500, episode_reward=19.61 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1124000, episode_reward=19.56 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1124500, episode_reward=19.49 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1125000, episode_reward=19.60 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1125500, episode_reward=19.66 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1126000, episode_reward=19.42 +/- 0.20
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1126500, episode_reward=19.55 +/- 0.10
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1127000, episode_reward=19.44 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=11

Eval num_timesteps=1168000, episode_reward=19.53 +/- 0.09
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1168500, episode_reward=19.57 +/- 0.09
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1169000, episode_reward=19.58 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1169500, episode_reward=19.67 +/- 0.08
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1170000, episode_reward=19.60 +/- 0.02
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1170500, episode_reward=19.53 +/- 0.02
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1171000, episode_reward=19.56 +/- 0.16
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1171500, episode_reward=19.57 +/- 0.09
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1172000, episode_reward=19.46 +/- 0.08
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1172500, episode_reward=19.44 +/- 0.15
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1173000, episode_reward=19.65 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=11

Eval num_timesteps=1214500, episode_reward=19.60 +/- 0.09
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1215000, episode_reward=19.64 +/- 0.10
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1215500, episode_reward=19.58 +/- 0.02
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1216000, episode_reward=19.48 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1216500, episode_reward=19.55 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1217000, episode_reward=19.50 +/- 0.02
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1217500, episode_reward=19.41 +/- 0.05
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1218000, episode_reward=19.46 +/- 0.07
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1218500, episode_reward=19.63 +/- 0.02
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1219000, episode_reward=19.51 +/- 0.09
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1219500, episode_reward=19.57 +/- 0.09
Episode length: 20.00 +/- 0.00
Eval num_timesteps=12

Eval num_timesteps=1261000, episode_reward=19.55 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1261500, episode_reward=19.61 +/- 0.03
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1262000, episode_reward=19.53 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1262500, episode_reward=19.52 +/- 0.01
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1263000, episode_reward=19.57 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1263500, episode_reward=19.54 +/- 0.08
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1264000, episode_reward=19.61 +/- 0.03
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1264500, episode_reward=19.50 +/- 0.08
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1265000, episode_reward=19.58 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1265500, episode_reward=19.57 +/- 0.04
Episode length: 20.00 +/- 0.00
Eval num_timesteps=1266000, episode_reward=19.64 +/- 0.06
Episode length: 20.00 +/- 0.00
Eval num_timesteps=12

In [9]:
SAC_path = os.path.join('Training', 'Saved Models', 'Elasticadesk10_SAC')   # Keep all the files in same directory
os.makedirs(os.path.dirname(SAC_path), exist_ok=True)

In [20]:
model.save(SAC_path)

In [11]:
model=SAC.load(SAC_path)
print(f"Model loaded from: {SAC_path}")

Model loaded from: Training\Saved Models\Elasticadesk10_SAC


C:\Users\Jayhind\anaconda3\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code() argument 13 must be str, not int
  warnings.warn(


In [13]:
env.enable_render = True
episodes = 50
for episode in range(1, episodes+1):
    state  , info = env.reset()
    done = False
    score = 0 
    truncation=False
    test=[]
    n_score=[]
    dis = []
    while not (done or truncation):
        action,_ = model.predict(state , deterministic=True)
        #print(action)
        state, reward, done, truncation ,info = env.step(action)
        n_score.append(reward)
        dis.append(state[-1])
        score+=reward
        test.append(done)
    #print(test)
    print(len(test))
    print(n_score)
    #print(np.min(dis))
    print('Episode:{} Score:{}'.format(episode, score))
env.close()

20
[0.9841800987343102, 0.9722122334181376, 0.9714057759267924, 0.9707886705672715, 0.9700385336480531, 0.9696135924390574, 0.9693888786544367, 0.969268208027326, 0.9692024027061407, 0.9691660505331223, 0.9691457991113327, 0.9691345094856796, 0.9691280766386786, 0.9691242795425161, 0.9691222372843701, 0.9691212083929951, 0.969120505281621, 0.9691201854679462, 0.9691199808208651, 0.9691198655249181]
Episode:1 Score:19.40652109220557
20
[0.981864492349309, 0.9755621216082059, 0.9736622156425753, 0.9732152713184725, 0.9731293897700748, 0.9731156579997257, 0.973115460066436, 0.9731180090192989, 0.9731208339707286, 0.9731239702802821, 0.973126670397006, 0.9731291107940904, 0.97313144925767, 0.9731332534768984, 0.9731349548452588, 0.9731362137787397, 0.9731375736424793, 0.9731386272663415, 0.9731393429043962, 0.973140055580633]
Episode:2 Score:19.474374673968622
20
[0.9783310444199383, 0.9684918121631073, 0.9684842686526652, 0.9684477956420706, 0.9683141807940663, 0.9679411186592167, 0.96765

In [22]:
env.close()